# Stage 3 — DPO Preference Alignment

Resumes from the Stage 2 SFT adapter and aligns it with `data/preference_dataset.jsonl` (prompt/chosen/rejected triples) using Direct Preference Optimization, so the model learns to prefer the correct, professional, domain-specific answer over a weak/rude/wrong one.

This notebook auto-generates `reports/final_evaluation.md` (base vs. SFT vs. DPO, three-way comparison).

Run on a GPU runtime (e.g. Google Colab).


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth trl


## 2. Shared setup

In [ ]:
import sys, os, json, gc
from pathlib import Path

sys.path.append("../src")
from report_utils import fill_report_section, make_markdown_table

EVAL_QUESTIONS = json.loads(Path("../data/eval_questions.json").read_text(encoding="utf-8"))

PROMPT_TEMPLATE = (
    "Below is a customer support question. Write a helpful, professional, "
    "domain-specific response.\n\n### Question:\n{}\n\n### Response:\n"
)


def ask(model, tokenizer, question, max_new_tokens=150):
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)
    prompt = PROMPT_TEMPLATE.format(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, use_cache=True,
        do_sample=False, temperature=1.0,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()


## 3. Load the model — resume from Stage 2 if available

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024
STAGE2_ADAPTER = "../outputs/stage2_adapter"
BASE_MODEL = "unsloth/Qwen2.5-0.5B"

model_name = STAGE2_ADAPTER if os.path.exists(STAGE2_ADAPTER) else BASE_MODEL
resumed_from_stage2 = model_name == STAGE2_ADAPTER

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
if not resumed_from_stage2:
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = 16,
        lora_dropout = 0.05,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )
    print("No Stage 2 adapter found - starting DPO from a fresh LoRA on the plain base model.")
else:
    print("Resumed from Stage 2 SFT adapter.")


## 4. Load and format the preference dataset

In [ ]:
from datasets import Dataset

pairs = []
with open("../data/preference_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            pairs.append(json.loads(line))
print(f"{len(pairs)} preference triples loaded")


def format_pref(ex):
    return {
        "prompt": PROMPT_TEMPLATE.format(ex["prompt"]),
        "chosen": ex["chosen"],
        "rejected": ex["rejected"],
    }


preference_dataset = Dataset.from_list([format_pref(p) for p in pairs])
preference_dataset[0]


## 5. Run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 5e-6,
    logging_steps = 5,
    optim = "adamw_8bit",
    lr_scheduler_type = "linear",
    seed = 42,
    beta = 0.1,
    max_length = max_seq_length,
    max_prompt_length = 512,
    output_dir = "../outputs/stage3_checkpoints",
    report_to = "none",
)

# With a PEFT/LoRA model and ref_model=None, TRL uses the frozen base
# weights (adapter disabled) as the implicit reference model - no need to
# load a second full copy of the model.
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = dpo_config,
    train_dataset = preference_dataset,
    tokenizer = tokenizer,  # older/newer TRL: rename to `processing_class` if this errors
)
dpo_stats = dpo_trainer.train()


## 6. Save the Stage 3 DPO-aligned model

In [ ]:
os.makedirs("../outputs", exist_ok=True)
model.save_pretrained("../outputs/stage3_dpo_model")
tokenizer.save_pretrained("../outputs/stage3_dpo_model")
print("Saved Stage 3 DPO-aligned model to ../outputs/stage3_dpo_model")


## 7. Quick sanity test after DPO

In [ ]:
for q in EVAL_QUESTIONS[:3]:
    print("Q:", q)
    print("A:", ask(model, tokenizer, q))
    print("---")


## 8. Final three-way evaluation -> `reports/final_evaluation.md`

Loads the base model, the Stage 2 SFT model, and the Stage 3 DPO model **independently** (freeing GPU memory between loads) so the comparison is a fair, from-scratch read of each stage rather than reused in-memory state.


In [ ]:
def generate_all(model_name, questions, base_model_for_adapter=None):
    from unsloth import FastLanguageModel
    import torch

    m, tok = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = True,
    )
    answers = [ask(m, tok, q) for q in questions]
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    return answers


base_answers = generate_all(BASE_MODEL, EVAL_QUESTIONS)
sft_answers = (
    generate_all(STAGE2_ADAPTER, EVAL_QUESTIONS)
    if os.path.exists(STAGE2_ADAPTER)
    else ["(no Stage 2 adapter found)"] * len(EVAL_QUESTIONS)
)
dpo_answers = generate_all("../outputs/stage3_dpo_model", EVAL_QUESTIONS)


In [ ]:
rows = [
    (q, b, s, d, "DPO", "Best balance of correctness, helpfulness and professional tone; review and edit this by hand after reading the actual answers.")
    for q, b, s, d in zip(EVAL_QUESTIONS, base_answers, sft_answers, dpo_answers)
]
table = make_markdown_table(
    ["Question", "Base Model Answer", "SFT Model Answer", "DPO Model Answer", "Best Answer", "Reason"],
    rows,
)
fill_report_section("../reports/final_evaluation.md", "final_eval", table)
print("Updated reports/final_evaluation.md")


## Done

All three stages are trained and saved under `outputs/` (git-ignored — see `.gitignore`). Use `src/inference.py` to query the final DPO-aligned model from the command line.
